# 08 — Remaining Trace Prediction

Multiclass classification: given a prefix of k events, predict the **entire suffix sequence** (all remaining activities).  
Each unique suffix string is treated as a separate class; rare suffixes (outside top-200 by training frequency) are mapped to `<OTHER>`.  
Methodology follows Teinemaa et al. (2019): prefix-length bucketing × Boolean/Succession encodings × LightGBM sweep.  
Metrics: Accuracy, Macro-F1, and Damerau-Levenshtein Similarity (DLS).

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os, random
import pm4py as pm
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, average_precision_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import math

TOP_K_SUFFIXES = 200  # number of most frequent suffix classes kept

## 2. Load & Clean

In [ ]:
dtype_dict = {
    'hired': 'Int8', 'Rejected': 'Int8', 'CW': 'Int8', 'Evergreen': 'Int8',
    'Region': 'Int16', 'Country': 'Int16',
    'Job Family': 'Int16', 'Job Family Group': 'Int16',
}
df = pd.read_csv('data/'+
    'event_log_consolidated.csv',
    low_memory=False,
    dtype=dtype_dict,
    parse_dates=['timestamp']
)
df = df[~df['Step'].str.match(r'^\d+$', na=False)].copy()
df.sort_values(['Case_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Shape: {df.shape}')
print(f'Cases: {df["Case_id"].nunique():,}')

## 3. Format for pm4py + Chronological 60/20/20 Split

In [ ]:
df = pm.format_dataframe(
    df, case_id='Case_id', activity_key='Step', timestamp_key='timestamp'
)

case_start = df.groupby('case:concept:name')['time:timestamp'].min().sort_values()
n_cases      = len(case_start)
train_cutoff = case_start.iloc[int(n_cases * 0.60)]
val_cutoff   = case_start.iloc[int(n_cases * 0.80)]

train_cases = case_start[case_start <  train_cutoff].index
val_cases   = case_start[(case_start >= train_cutoff) & (case_start < val_cutoff)].index
test_cases  = case_start[case_start >= val_cutoff].index

print(f'Train: {len(train_cases):,} | Val: {len(val_cases):,} | Test: {len(test_cases):,}')
print(f'Train cutoff: {train_cutoff.date()} | Val cutoff: {val_cutoff.date()}')

train_df = df[df['case:concept:name'].isin(train_cases)].copy()
val_df   = df[df['case:concept:name'].isin(val_cases)].copy()
test_df  = df[df['case:concept:name'].isin(test_cases)].copy()

## 4. Remove Leaky Activities + Build Activity Vocabulary

In [ ]:
LEAKY_THRESHOLD = 0.85
act_hire_rate = (
    train_df.groupby('concept:name')['hired']
    .apply(lambda x: x.astype(float).mean())
    .sort_values(ascending=False)
)
leaky_acts = set(act_hire_rate[act_hire_rate >= LEAKY_THRESHOLD].index)
print(f'Leaky activities removed ({len(leaky_acts)}):')
for act in sorted(leaky_acts, key=lambda a: -act_hire_rate[a]):
    print(f'  {act_hire_rate[act]:.3f}  {act}')

train_df = train_df[~train_df['concept:name'].isin(leaky_acts)].copy()
val_df   = val_df[~val_df['concept:name'].isin(leaky_acts)].copy()
test_df  = test_df[~test_df['concept:name'].isin(leaky_acts)].copy()

all_train_acts = sorted(train_df['concept:name'].dropna().unique())
act2idx = {act: i+2 for i, act in enumerate(all_train_acts)}
act2idx['<PAD>'] = 0
act2idx['<UNK>'] = 1
idx2act = {v: k for k, v in act2idx.items()}
VOCAB_SIZE  = len(act2idx)
MAX_SEQ_LEN = 30
print(f'\nVocabulary size: {VOCAB_SIZE} | MAX_SEQ_LEN: {MAX_SEQ_LEN}')

## 5. Encoding Helpers (Boolean / Succession)

In [ ]:
PM_COLS        = ['case:concept:name', 'concept:name', 'time:timestamp']
SKIP_COLS      = {'case:concept:name', 'concept:name', 'time:timestamp', '@@diff_start_end'}
CASE_ATTR_COLS = ['Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen']

case_attrs_train = train_df.groupby('case:concept:name')[CASE_ATTR_COLS].first().fillna(0).astype(float)
case_attrs_val   = val_df.groupby('case:concept:name')[CASE_ATTR_COLS].first().fillna(0).astype(float)
case_attrs_test  = test_df.groupby('case:concept:name')[CASE_ATTR_COLS].first().fillna(0).astype(float)


def get_prefix_pm(df_clean, k):
    if k == 'all':
        return df_clean[PM_COLS].copy()
    prefix = pm.get_prefixes_from_log(df_clean, length=k, case_id_key='case:concept:name')
    return prefix[PM_COLS].copy()


def bool_encoding(prefix_pm):
    enriched = pm.extract_outcome_enriched_dataframe(
        prefix_pm, activity_key='concept:name',
        timestamp_key='time:timestamp', case_id_key='case:concept:name',
    )
    feat_cols = [c for c in enriched.columns if c not in SKIP_COLS]
    return enriched.drop_duplicates('case:concept:name').set_index('case:concept:name')[feat_cols].sort_index()


def succ_encoding(prefix_pm):
    all_case_ids = sorted(prefix_pm['case:concept:name'].unique())
    df_s = prefix_pm.copy()
    df_s['prev_act'] = df_s.groupby('case:concept:name')['concept:name'].shift(1)
    df_s = df_s.dropna(subset=['prev_act'])
    if df_s.empty:
        return pd.DataFrame(index=pd.Index(all_case_ids, name='case:concept:name'))
    df_s['bigram'] = df_s['prev_act'] + ' >> ' + df_s['concept:name']
    bigram_counts = df_s.groupby(['case:concept:name', 'bigram']).size().unstack(fill_value=0)
    return bigram_counts.reindex(all_case_ids, fill_value=0).rename_axis('case:concept:name')


def build_Xy(X_pm, case_attrs_split, y_series, train_cols=None):
    if train_cols is not None:
        X_pm = X_pm.reindex(columns=train_cols, fill_value=0)
    X = X_pm.join(case_attrs_split, how='left').fillna(0)
    common = X.index.intersection(y_series.index)
    X = X.loc[common]
    y = y_series.reindex(common).values
    return X.values.astype(np.float32), y, list(X.columns)


print('Encoding helpers defined.')

## 6. Suffix Label Extraction

For prefix of length k, the suffix is the sequence of activities from position k onward, joined by `' | '`.  
The top-`TOP_K_SUFFIXES` suffixes (by training frequency) form the label vocabulary; all others → `<OTHER>`.

In [ ]:
OTHER_LABEL = '<OTHER>'


def extract_suffix_strings(df_subset, k):
    """Return Series: case_id -> suffix string (activities from position k onward)."""
    records = {}
    for cid, grp in df_subset.groupby('case:concept:name', sort=False):
        grp = grp.sort_values('time:timestamp')
        if len(grp) <= k:
            continue  # no suffix
        suffix_acts = grp.iloc[k:]['concept:name'].tolist()
        records[cid] = ' | '.join(suffix_acts)
    return pd.Series(records, name='suffix')


def build_suffix_vocab(train_suffix_series, top_k=TOP_K_SUFFIXES):
    """Build top-k suffix vocabulary from training labels."""
    counts = train_suffix_series.value_counts()
    top_suffixes = list(counts.head(top_k).index)
    suffix_set = set(top_suffixes)
    print(f'Suffix vocab: {len(top_suffixes)} classes cover '
          f'{counts.head(top_k).sum()/len(train_suffix_series)*100:.1f}% of train cases')
    return suffix_set


def remap_suffixes(suffix_series, suffix_set):
    """Replace rare suffixes with OTHER_LABEL."""
    return suffix_series.map(lambda s: s if s in suffix_set else OTHER_LABEL)


def damerau_levenshtein_dist(s1, s2):
    """Damerau-Levenshtein distance on activity lists."""
    a, b = s1.split(' | ') if s1 != OTHER_LABEL else [OTHER_LABEL], \
           s2.split(' | ') if s2 != OTHER_LABEL else [OTHER_LABEL]
    n, m = len(a), len(b)
    d = np.zeros((n+1, m+1), dtype=int)
    for i in range(n+1): d[i, 0] = i
    for j in range(m+1): d[0, j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            d[i, j] = min(d[i-1, j]+1, d[i, j-1]+1, d[i-1, j-1]+cost)
            if i > 1 and j > 1 and a[i-1] == b[j-2] and a[i-2] == b[j-1]:
                d[i, j] = min(d[i, j], d[i-2, j-2]+cost)
    return d[n, m]


def dls_score(y_true_labels, y_pred_labels, max_sample=1000):
    """Mean DLS similarity (1 - normalised DL distance) on a random sample."""
    idx = np.random.choice(len(y_true_labels), size=min(max_sample, len(y_true_labels)), replace=False)
    sims = []
    for i in idx:
        s1, s2 = y_true_labels[i], y_pred_labels[i]
        dist = damerau_levenshtein_dist(s1, s2)
        max_len = max(len(s1.split(' | ')), len(s2.split(' | ')))
        sims.append(1.0 - dist / max(max_len, 1))
    return float(np.mean(sims))


print('Label extraction and DLS helpers defined.')

## 7. Prefix-Length Sweep — LightGBM (Boolean × Succession)

For each (encoding, k) pair: fit LightGBM multiclass with early stopping on the validation set.  
Metrics: Accuracy and Macro-F1.

In [ ]:
PREFIX_LENGTHS = [1, 3, 5, 10, 20]
ENCODINGS      = [('Boolean', bool_encoding), ('Succession', succ_encoding)]
sweep_results  = {}
suffix_vocabs  = {}  # k -> suffix_set (built on train)
label_encoders = {}  # k -> LabelEncoder fitted on (suffix_set ∪ {OTHER_LABEL})

for enc_name, enc_fn in ENCODINGS:
    print(f'\n── {enc_name} encoding ──────────────────────')
    for k in PREFIX_LENGTHS:
        # ---- Suffix labels ----
        suf_tr_raw = extract_suffix_strings(train_df, k)
        suf_va_raw = extract_suffix_strings(val_df,   k)
        suf_te_raw = extract_suffix_strings(test_df,  k)

        suffix_set = build_suffix_vocab(suf_tr_raw, top_k=TOP_K_SUFFIXES)
        suffix_vocabs[k] = suffix_set

        suf_tr = remap_suffixes(suf_tr_raw, suffix_set)
        suf_va = remap_suffixes(suf_va_raw, suffix_set)
        suf_te = remap_suffixes(suf_te_raw, suffix_set)

        # LabelEncoder fitted only on labels seen in training (suffix_set + OTHER)
        le = LabelEncoder().fit(list(suffix_set) + [OTHER_LABEL])
        label_encoders[k] = le

        # ---- Features ----
        train_prefix_pm = get_prefix_pm(train_df, k)
        val_prefix_pm   = get_prefix_pm(val_df,   k)
        test_prefix_pm  = get_prefix_pm(test_df,  k)

        X_tr_pm    = enc_fn(train_prefix_pm)
        train_cols = list(X_tr_pm.columns)
        X_va_pm    = enc_fn(val_prefix_pm)
        X_te_pm    = enc_fn(test_prefix_pm)

        X_tr, y_tr_f, _ = build_Xy(X_tr_pm, case_attrs_train, suf_tr)
        X_va, y_va_f, _ = build_Xy(X_va_pm, case_attrs_val,   suf_va, train_cols=train_cols)
        X_te, y_te_f, _ = build_Xy(X_te_pm, case_attrs_test,  suf_te, train_cols=train_cols)

        y_tr_enc = le.transform(y_tr_f)
        y_va_enc = le.transform(y_va_f)
        y_te_enc = le.transform(y_te_f)

        if len(X_tr) == 0 or len(X_te) == 0:
            print(f'  k={k}: skipping (empty split)')
            continue

        # Val may contain suffix labels remapped to OTHER that still weren't seen in train's
        # encoded set at this prefix length; filter eval_set to safe labels only.
        train_label_set = set(y_tr_enc.tolist())
        va_mask = np.isin(y_va_enc, list(train_label_set))
        X_va_es, y_va_es = X_va[va_mask], y_va_enc[va_mask]

        clf = LGBMClassifier(
            n_estimators=300, class_weight='balanced',
            n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
        )
        clf.fit(
            X_tr, y_tr_enc,
            eval_set=[(X_va_es, y_va_es)],
            callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(period=-1)],
        )
        pred_enc    = clf.predict(X_te)
        proba       = clf.predict_proba(X_te)
        pred_labels = le.inverse_transform(pred_enc)
        true_labels = le.inverse_transform(y_te_enc)

        acc  = accuracy_score(y_te_enc, pred_enc)
        f1   = f1_score(y_te_enc, pred_enc, average='macro', zero_division=0)
        dls  = dls_score(list(true_labels), list(pred_labels))
        # Macro AUC-PR: one-vs-rest average precision over all suffix classes
        ap   = average_precision_score(y_te_enc, proba, average='macro')
        sweep_results[(enc_name, k)] = {'Acc': acc, 'F1': f1, 'DLS': dls, 'AP': ap}
        print(f'  k={str(k):3s}  Acc={acc:.4f}  Macro-F1={f1:.4f}  DLS={dls:.4f}  AUC-PR={ap:.4f}')

## 8. Full-Trace Models (k = 20)

RF / XGBoost / LightGBM trained with Boolean + Succession encodings at k=20.

In [ ]:
k_full = 20

suf_tr_raw_f = extract_suffix_strings(train_df, k_full)
suf_va_raw_f = extract_suffix_strings(val_df,   k_full)
suf_te_raw_f = extract_suffix_strings(test_df,  k_full)

suffix_set_f = build_suffix_vocab(suf_tr_raw_f, top_k=TOP_K_SUFFIXES)
suf_tr_f = remap_suffixes(suf_tr_raw_f, suffix_set_f)
suf_va_f = remap_suffixes(suf_va_raw_f, suffix_set_f)
suf_te_f = remap_suffixes(suf_te_raw_f, suffix_set_f)
le_f = LabelEncoder().fit(list(suffix_set_f) + [OTHER_LABEL])

# Boolean
X_bool_tr_pm = bool_encoding(get_prefix_pm(train_df, k_full))
bool_cols    = list(X_bool_tr_pm.columns)
X_bool_tr, y_tr_raw, _ = build_Xy(X_bool_tr_pm, case_attrs_train, suf_tr_f)
X_bool_va, y_va_raw, _ = build_Xy(bool_encoding(get_prefix_pm(val_df,   k_full)), case_attrs_val,   suf_va_f, bool_cols)
X_bool_te, y_te_raw, _ = build_Xy(bool_encoding(get_prefix_pm(test_df,  k_full)), case_attrs_test,  suf_te_f, bool_cols)
y_tr = le_f.transform(y_tr_raw); y_va = le_f.transform(y_va_raw); y_te = le_f.transform(y_te_raw)

# Succession
X_succ_tr_pm = succ_encoding(get_prefix_pm(train_df, k_full))
succ_cols    = list(X_succ_tr_pm.columns)
X_succ_tr, _, _ = build_Xy(X_succ_tr_pm, case_attrs_train, suf_tr_f)
X_succ_va, _, _ = build_Xy(succ_encoding(get_prefix_pm(val_df,   k_full)), case_attrs_val,   suf_va_f, succ_cols)
X_succ_te, _, _ = build_Xy(succ_encoding(get_prefix_pm(test_df,  k_full)), case_attrs_test,  suf_te_f, succ_cols)

print(f'Boolean:    train={X_bool_tr.shape}, test={X_bool_te.shape}')
print(f'Succession: train={X_succ_tr.shape}, test={X_succ_te.shape}')
print(f'Classes: {len(le_f.classes_)}')

In [ ]:
full_results = {}


def run_clf(X_train, X_val, X_test, y_train, y_val, y_test, label, le):
    models = {
        'RF':  RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                      n_jobs=-1, random_state=RANDOM_STATE),
        'XGB': XGBClassifier(n_estimators=300, objective='multi:softmax',
                             n_jobs=-1, random_state=RANDOM_STATE, verbosity=0,
                             early_stopping_rounds=20, eval_metric='mlogloss'),
        'LGBM': LGBMClassifier(n_estimators=300, class_weight='balanced',
                               n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
    }
    res = {}
    for name, clf in models.items():
        print(f'  [{label}] {name}...')
        if name == 'XGB':
            clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        elif name == 'LGBM':
            clf.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                    callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(period=-1)])
        else:
            clf.fit(X_train, y_train)
        pred_enc    = clf.predict(X_test)
        proba       = clf.predict_proba(X_test)
        pred_labels = le.inverse_transform(pred_enc)
        true_labels = le.inverse_transform(y_test)
        acc  = accuracy_score(y_test, pred_enc)
        f1   = f1_score(y_test, pred_enc, average='macro', zero_division=0)
        dls  = dls_score(list(true_labels), list(pred_labels))
        ap   = average_precision_score(y_test, proba, average='macro')
        res[name] = {'Acc': acc, 'Macro-F1': f1, 'DLS': dls, 'AUC-PR': ap}
        print(f'    Acc={acc:.4f}  Macro-F1={f1:.4f}  DLS={dls:.4f}  AUC-PR={ap:.4f}')
    return res


print('=== Boolean Encoding (k=20) ===')
full_results['Boolean']    = run_clf(X_bool_tr, X_bool_va, X_bool_te, y_tr, y_va, y_te, 'Boolean', le_f)
print('\n=== Succession Encoding (k=20) ===')
full_results['Succession'] = run_clf(X_succ_tr, X_succ_va, X_succ_te, y_tr, y_va, y_te, 'Succession', le_f)

## 9. Deep Learning — LSTM + Transformer Classifier

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(torch.__version__)

N_CLASSES = len(le_f.classes_)


def build_sequences(df_subset, case_ids, k, act2idx, max_seq_len):
    seqs = []
    df_s = df_subset.sort_values(['case:concept:name', 'time:timestamp'])
    grp_map = {cid: grp for cid, grp in df_s.groupby('case:concept:name', sort=False)}
    for cid in case_ids:
        grp = grp_map.get(cid)
        if grp is None:
            seqs.append([0] * max_seq_len)
            continue
        acts = grp['concept:name'].tolist()
        prefix_acts = acts[:k] if k != 'all' else acts
        idxs = [act2idx.get(a, act2idx['<UNK>']) for a in prefix_acts]
        seq  = idxs[-max_seq_len:]
        seqs.append([0] * (max_seq_len - len(seq)) + seq)
    return np.array(seqs, dtype=np.int64)


seqs_tr = build_sequences(train_df, suf_tr_f.index, k_full, act2idx, MAX_SEQ_LEN)
seqs_va = build_sequences(val_df,   suf_va_f.index, k_full, act2idx, MAX_SEQ_LEN)
seqs_te = build_sequences(test_df,  suf_te_f.index, k_full, act2idx, MAX_SEQ_LEN)

scaler = MinMaxScaler()
X_tr_tab = scaler.fit_transform(X_bool_tr).astype(np.float32)
X_va_tab = scaler.transform(X_bool_va).astype(np.float32)
X_te_tab = scaler.transform(X_bool_te).astype(np.float32)
N_TAB = X_tr_tab.shape[1]


class RemTraceDataset(Dataset):
    def __init__(self, X, seqs, y):
        self.X    = torch.tensor(X,    dtype=torch.float32)
        self.seqs = torch.tensor(seqs, dtype=torch.long)
        self.y    = torch.tensor(y,    dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.seqs[i], self.y[i]


train_ds = RemTraceDataset(X_tr_tab, seqs_tr, y_tr)
val_ds   = RemTraceDataset(X_va_tab, seqs_va, y_va)
test_ds  = RemTraceDataset(X_te_tab, seqs_te, y_te)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')

In [ ]:
class LSTMRemTrace(nn.Module):
    def __init__(self, vocab_size, n_classes, emb_dim=64, hidden=128, n_layers=2, dropout=0.3, n_tab=10):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden, num_layers=n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.fc1  = nn.Linear(hidden + n_tab, 128)
        self.fc2  = nn.Linear(128, n_classes)
        self.drop = nn.Dropout(dropout)

    def forward(self, seq, tab):
        x = self.emb(seq)
        _, (h, _) = self.lstm(x)
        combined = torch.cat([h[-1], tab], dim=1)
        return self.fc2(torch.relu(self.fc1(self.drop(combined))))


def train_clf_model(model, train_loader, val_loader, n_epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    best_acc, best_state, no_improve = 0.0, None, 0

    for epoch in range(n_epochs):
        model.train()
        for X_b, seq_b, y_b in train_loader:
            X_b, seq_b, y_b = X_b.to(device), seq_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            criterion(model(seq_b, X_b), y_b).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        preds_v, targets_v = [], []
        with torch.no_grad():
            for X_b, seq_b, y_b in val_loader:
                X_b, seq_b = X_b.to(device), seq_b.to(device)
                preds_v.extend(model(seq_b, X_b).argmax(1).cpu().numpy())
                targets_v.extend(y_b.numpy())
        val_acc = accuracy_score(targets_v, preds_v)
        scheduler.step(-val_acc)
        print(f'  Epoch {epoch+1:02d}  val_Acc={val_acc:.4f}')

        if val_acc > best_acc:
            best_acc   = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= 5:
                print(f'  Early stop at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return model


lstm_model = LSTMRemTrace(VOCAB_SIZE, N_CLASSES, n_tab=N_TAB).to(device)
print('Training LSTM...')
lstm_model = train_clf_model(lstm_model, train_loader, val_loader)

lstm_model.eval()
all_preds, all_probs, all_targets = [], [], []
with torch.no_grad():
    for X_b, seq_b, y_b in test_loader:
        X_b, seq_b = X_b.to(device), seq_b.to(device)
        logits = lstm_model(seq_b, X_b)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_probs.extend(torch.softmax(logits, dim=1).cpu().numpy())
        all_targets.extend(y_b.numpy())
all_probs = np.array(all_probs)
lstm_acc = accuracy_score(all_targets, all_preds)
lstm_f1  = f1_score(all_targets, all_preds, average='macro', zero_division=0)
lstm_dls = dls_score(list(le_f.inverse_transform(all_targets)),
                     list(le_f.inverse_transform(all_preds)))
lstm_ap  = average_precision_score(all_targets, all_probs, average='macro')
print(f'LSTM  Acc={lstm_acc:.4f}  Macro-F1={lstm_f1:.4f}  DLS={lstm_dls:.4f}  AUC-PR={lstm_ap:.4f}')

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerRemTrace(nn.Module):
    def __init__(self, vocab_size, n_classes, emb_dim=64, nhead=4, n_layers=2,
                 d_ff=128, dropout=0.1, n_tab=10):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.pe  = PositionalEncoding(emb_dim, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=nhead,
                                               dim_feedforward=d_ff, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.fc1 = nn.Linear(emb_dim + n_tab, 128)
        self.fc2 = nn.Linear(128, n_classes)
        self.drop = nn.Dropout(dropout)

    def forward(self, seq, tab):
        pad_mask = (seq == 0)
        x = self.pe(self.emb(seq))
        x = self.encoder(x, src_key_padding_mask=pad_mask)
        mask_f = (~pad_mask).unsqueeze(-1).float()
        x = (x * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)
        return self.fc2(torch.relu(self.fc1(self.drop(torch.cat([x, tab], dim=1)))))


tf_model = TransformerRemTrace(VOCAB_SIZE, N_CLASSES, n_tab=N_TAB).to(device)
print('Training Transformer...')
tf_model = train_clf_model(tf_model, train_loader, val_loader)

tf_model.eval()
all_preds, all_probs, all_targets = [], [], []
with torch.no_grad():
    for X_b, seq_b, y_b in test_loader:
        X_b, seq_b = X_b.to(device), seq_b.to(device)
        logits = tf_model(seq_b, X_b)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_probs.extend(torch.softmax(logits, dim=1).cpu().numpy())
        all_targets.extend(y_b.numpy())
all_probs = np.array(all_probs)
tf_acc = accuracy_score(all_targets, all_preds)
tf_f1  = f1_score(all_targets, all_preds, average='macro', zero_division=0)
tf_dls = dls_score(list(le_f.inverse_transform(all_targets)),
                   list(le_f.inverse_transform(all_preds)))
tf_ap  = average_precision_score(all_targets, all_probs, average='macro')
print(f'Transformer  Acc={tf_acc:.4f}  Macro-F1={tf_f1:.4f}  DLS={tf_dls:.4f}  AUC-PR={tf_ap:.4f}')

## 10. Results Table + Metrics vs Prefix Plot

In [ ]:
print('\n=== SWEEP RESULTS ===')
for (enc, k), m in sorted(sweep_results.items(), key=lambda x: (x[0][0], x[0][1])):
    print(f'{enc:12s}  k={str(k):3s}  Acc={m["Acc"]:.4f}  Macro-F1={m["F1"]:.4f}  '
          f'DLS={m["DLS"]:.4f}  AUC-PR={m["AP"]:.4f}')

print('\n=== FULL-TRACE MODELS (k=20) ===')
for enc, res in full_results.items():
    for name, m in res.items():
        print(f'{enc:12s}  {name:6s}  Acc={m["Acc"]:.4f}  Macro-F1={m["Macro-F1"]:.4f}  '
              f'DLS={m["DLS"]:.4f}  AUC-PR={m["AUC-PR"]:.4f}')

print(f'\nLSTM         Acc={lstm_acc:.4f}  Macro-F1={lstm_f1:.4f}  DLS={lstm_dls:.4f}  AUC-PR={lstm_ap:.4f}')
print(f'Transformer  Acc={tf_acc:.4f}  Macro-F1={tf_f1:.4f}  DLS={tf_dls:.4f}  AUC-PR={tf_ap:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
metrics = [('Acc', 'Accuracy'), ('F1', 'Macro-F1'), ('DLS', 'DLS Similarity'), ('AP', 'AUC-PR (macro)')]
for ax, (mk, title) in zip(axes, metrics):
    for enc_name, _ in ENCODINGS:
        vals = [sweep_results.get((enc_name, k), {}).get(mk, np.nan) for k in PREFIX_LENGTHS]
        ax.plot([str(k) for k in PREFIX_LENGTHS], vals, marker='o', label=enc_name)
    ax.set_title(f'Remaining Trace: {title} vs Prefix Length')
    ax.set_xlabel('Prefix Length'); ax.set_ylabel(title); ax.legend()
plt.tight_layout()
plt.savefig('figs/remaining_trace_metrics_vs_prefix.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved figs/remaining_trace_metrics_vs_prefix.png')